<a href="https://colab.research.google.com/github/YibenZhou25/demo/blob/main/gradpath_main_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GradPath Dublin · Career-Readiness Agent

**COMP47980 · Generative AI and Language Models · Project 2026**

An LLM-based assistant for international Master's-level Computer Science students preparing for graduate / junior software engineering roles in Dublin.

## What this notebook does

This notebook builds and runs a domain-specific agent on top of OpenAI's Responses API. The agent uses all three Responses-API tools — **File Search**, **Function Calling**, and **Code Interpreter** — and integrates **three external APIs**: GitHub REST, Greenhouse Job Boards, and The Muse.

Run cells in order. Sections are clearly marked.

## Prerequisites

Before running, configure these Colab secrets (🔑 icon, left sidebar):

- `OPENAI_API_KEY` — your OpenAI API key
- `GITHUB_TOKEN` — a GitHub personal access token (scope: public_repo, optional but increases rate limit)

Greenhouse and The Muse are used without authentication — both are public APIs.

## Cost estimate

Running this notebook end-to-end including all worked examples typically costs under €2 in OpenAI usage when using `gpt-4.1`.

---
## Section 1 · Setup

Install dependencies, mount Google Drive, configure clients.

In [4]:
!pip install openai --quiet

from google.colab import drive, userdata
from openai import OpenAI
import os, json, time, requests
from datetime import date, datetime
from pathlib import Path

# Mount Drive (knowledge-base files live here)
drive.mount('/content/drive')

# Load secrets — never put keys in source code
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
    print('⚠ No GITHUB_TOKEN found — GitHub API calls will use unauthenticated rate limit (60/hr)')

client = OpenAI()

# Knowledge base location
KB_ROOT = '/content/drive/MyDrive/gradpath_knowledge_base'
MODEL = 'gpt-4.1'  # primary model — chosen for cost/quality balance per project budget

print('✓ OpenAI client ready')
print(f'✓ Knowledge base root: {KB_ROOT}')
print(f'✓ Model: {MODEL}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ OpenAI client ready
✓ Knowledge base root: /content/drive/MyDrive/gradpath_knowledge_base
✓ Model: gpt-4.1


---
## Section 2 · Vector Store · File Search backbone

Upload the curated knowledge base into an OpenAI vector store. The agent retrieves chunks from this store at conversation time using the File Search tool.

**Note**: this section creates a vector store the first time it runs. On subsequent runs it reuses the existing store via the `VECTOR_STORE_ID` cached locally. To rebuild from scratch, delete the cache file.

In [5]:
VECTOR_STORE_CACHE = '/content/drive/MyDrive/gradpath_knowledge_base/.vector_store_id'

def collect_kb_files():
    """Walk the knowledge base and return all .md and .pdf files."""
    files = []
    valid_extensions = ('.md', '.pdf')
    for sub in ['00_meta', '01_companies', '02_interviews', '03_rubrics',
                '04_visa_and_market', '05_cultural']:
        sub_path = os.path.join(KB_ROOT, sub)
        if not os.path.isdir(sub_path):
            continue
        for fname in sorted(os.listdir(sub_path)):
            # Skip templates and hidden files
            if fname.startswith('_template') or fname.startswith('.'):
                continue
            if fname.endswith(valid_extensions):
                files.append(os.path.join(sub_path, fname))
    return files

def get_or_create_vector_store():
    """Create the vector store on first run; reuse on subsequent runs."""
    if os.path.exists(VECTOR_STORE_CACHE):
        with open(VECTOR_STORE_CACHE) as f:
            store_id = f.read().strip()
        try:
            client.vector_stores.retrieve(store_id)
            print(f'✓ Reusing existing vector store: {store_id}')
            return store_id
        except Exception:
            print('⚠ Cached store ID invalid, creating a new one')

    # Fresh creation
    print('Creating new vector store...')
    store = client.vector_stores.create(name='gradpath_dublin_kb')
    files = collect_kb_files()
    print(f'Uploading {len(files)} files...')
    file_ids = []
    for fp in files:
        with open(fp, 'rb') as f:
            f_obj = client.files.create(file=f, purpose='assistants')
            file_ids.append(f_obj.id)
        print(f'  ✓ {os.path.basename(fp)} → {f_obj.id}')
    client.vector_stores.file_batches.create(
        vector_store_id=store.id,
        file_ids=file_ids,
    )
    # Wait for indexing
    print('Waiting for indexing...')
    while True:
        s = client.vector_stores.retrieve(store.id)
        print(f'  status: {s.status}')
        if s.status == 'completed':
            break
        time.sleep(3)
    # Cache the ID
    with open(VECTOR_STORE_CACHE, 'w') as f:
        f.write(store.id)
    print(f'✓ Vector store ready: {store.id}')
    return store.id

VECTOR_STORE_ID = get_or_create_vector_store()

✓ Reusing existing vector store: vs_69f7348d21f0819198978a0ede4d4a6b


---
## Section 3 · External-API Function Implementations

These are the Python functions the agent will call when its reasoning concludes that one of the three external APIs is needed. Each function returns a JSON-serialisable dictionary.

Three external APIs are integrated:

1. **Greenhouse Job Boards API** — live Dublin tech jobs from canonical employer ATS
2. **GitHub REST API** — user's repository portfolio for evidence audit
3. **The Muse API** — supplemental company-culture data

Plus one local function:

4. **`check_visa_eligibility()`** — deterministic visa-threshold computation

In [6]:
# ----------------------------------------------------------
# External API #1 — Greenhouse Job Boards (live Dublin jobs)
# ----------------------------------------------------------
# Aggregates postings from a curated set of Dublin tech employers' ATS feeds.
# Replaces Adzuna (which lacks Ireland-region coverage on free tier).

GREENHOUSE_BOARDS = [
    'stripe', 'intercom', 'tines', 'squarespace', 'pinterest',
    'fivetran', 'datadog', 'mongodb', 'elastic', 'dropbox',
    'twilio', 'okta', 'klaviyo',
]

def search_dublin_jobs(role_keyword: str, max_results: int = 8) -> dict:
    """
    Search live Dublin tech jobs across curated Greenhouse boards.

    Args:
        role_keyword: keyword to filter titles (e.g. 'graduate', 'junior', 'data')
        max_results: cap on returned roles
    """
    matches = []
    role_lower = role_keyword.lower()
    for board in GREENHOUSE_BOARDS:
        try:
            url = f'https://boards-api.greenhouse.io/v1/boards/{board}/jobs'
            r = requests.get(url, timeout=10)
            if r.status_code != 200:
                continue
            for job in r.json().get('jobs', []):
                location = job.get('location', {}).get('name', '').lower()
                title = job.get('title', '')
                if 'dublin' in location or 'ireland' in location:
                    if role_lower in title.lower() or role_lower == '':
                        matches.append({
                            'company': board.title(),
                            'title': title,
                            'location': job.get('location', {}).get('name', ''),
                            'url': job.get('absolute_url', ''),
                            'updated_at': job.get('updated_at', ''),
                        })
        except Exception:
            continue
    return {
        'query': role_keyword,
        'fetched_at': datetime.utcnow().isoformat() + 'Z',
        'total_matches': len(matches),
        'jobs': matches[:max_results],
        'source': 'Greenhouse public Job Boards API across curated Dublin employers',
    }

# Quick smoke test
_test = search_dublin_jobs('engineer', max_results=3)
print(f"✓ Greenhouse: found {_test['total_matches']} matches, showing first 3")
for j in _test['jobs']:
    print(f"   - {j['company']}: {j['title']}")

✓ Greenhouse: found 125 matches, showing first 3
   - Stripe: Backend Engineer/API, Payments and Risk
   - Stripe: Backend Engineer, Core Technology
   - Stripe: Engineering Manager - Fraud Risk 


/tmp/ipykernel_27543/1091778175.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'fetched_at': datetime.utcnow().isoformat() + 'Z',


In [7]:
# ----------------------------------------------------------
# External API #2 — GitHub REST (user portfolio audit)
# ----------------------------------------------------------

def get_user_repos(username: str, max_repos: int = 10) -> dict:
    """
    Fetch a GitHub user's public repositories, sorted by recent activity.
    Returns metadata for portfolio-strength assessment.
    """
    headers = {'Accept': 'application/vnd.github+json'}
    if GITHUB_TOKEN:
        headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'
    url = f'https://api.github.com/users/{username}/repos'
    params = {'sort': 'updated', 'per_page': max_repos, 'type': 'owner'}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=10)
        r.raise_for_status()
        repos = r.json()
        summary = []
        for repo in repos[:max_repos]:
            summary.append({
                'name': repo.get('name'),
                'description': repo.get('description') or '(no description)',
                'language': repo.get('language'),
                'stars': repo.get('stargazers_count', 0),
                'forks': repo.get('forks_count', 0),
                'is_fork': repo.get('fork', False),
                'updated_at': repo.get('updated_at'),
                'url': repo.get('html_url'),
                'has_readme': True,  # to be confirmed by readme call below
            })
        return {
            'username': username,
            'total_repos_fetched': len(summary),
            'repos': summary,
            'source': 'GitHub REST API v3',
        }
    except Exception as e:
        return {'error': str(e), 'username': username, 'repos': []}

def get_repo_readme(username: str, repo_name: str) -> dict:
    """
    Fetch the README content of a specific repo for content-quality audit.
    Returns the first 2000 characters of decoded README, or an error.
    """
    import base64
    headers = {'Accept': 'application/vnd.github+json'}
    if GITHUB_TOKEN:
        headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'
    url = f'https://api.github.com/repos/{username}/{repo_name}/readme'
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 404:
            return {'username': username, 'repo': repo_name, 'has_readme': False,
                    'readme_excerpt': None}
        r.raise_for_status()
        data = r.json()
        content = base64.b64decode(data.get('content', '')).decode('utf-8', errors='ignore')
        return {
            'username': username,
            'repo': repo_name,
            'has_readme': True,
            'readme_size_chars': len(content),
            'readme_excerpt': content[:2000],
            'source': 'GitHub REST API v3',
        }
    except Exception as e:
        return {'error': str(e), 'username': username, 'repo': repo_name}

# Smoke test (pick a public account that exists, e.g. octocat)
_test = get_user_repos('octocat', max_repos=3)
print(f"✓ GitHub: fetched {_test.get('total_repos_fetched', 0)} repos for octocat")

✓ GitHub: fetched 3 repos for octocat


In [8]:
# ----------------------------------------------------------
# External API #3 — The Muse (company-culture insights)
# ----------------------------------------------------------

def get_company_insights(company_name: str) -> dict:
    """
    Fetch up-to-date culture / role data from The Muse.
    Supplements the static company profiles in our vector store with
    fresh job-posting and culture context.
    """
    url = 'https://www.themuse.com/api/public/jobs'
    params = {
        'company': company_name,
        'location': 'Dublin, Ireland',
        'page': 0,
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
        results = data.get('results', [])
        # Distill into a digestible summary
        roles = []
        for j in results[:5]:
            roles.append({
                'name': j.get('name'),
                'level': j.get('levels', [{}])[0].get('name') if j.get('levels') else None,
                'category': j.get('categories', [{}])[0].get('name') if j.get('categories') else None,
                'short_summary': (j.get('contents') or '')[:500].replace('<p>', '').replace('</p>', ' '),
                'url': j.get('refs', {}).get('landing_page'),
            })
        return {
            'company': company_name,
            'location_filter': 'Dublin, Ireland',
            'roles_found': len(results),
            'roles_sample': roles,
            'source': 'The Muse Public API',
        }
    except Exception as e:
        return {'error': str(e), 'company': company_name}

# Smoke test
_test = get_company_insights('Stripe')
print(f"✓ The Muse: found {_test.get('roles_found', 0)} role records for Stripe in Dublin")

✓ The Muse: found 0 role records for Stripe in Dublin


In [9]:
# ----------------------------------------------------------
# Local Function — Visa eligibility computation
# ----------------------------------------------------------
# Deterministic threshold logic, kept local because policy values are
# stable enough not to require live API and because precise computation
# is exactly what the LLM cannot do reliably on its own.

# Updated to reflect post-1-March-2026 thresholds (DETE Roadmap Review 2025)
VISA_THRESHOLDS_2026 = {
    'critical_skills_standard': 40904,        # Standard CSEP, CSOL role + degree
    'critical_skills_recent_grad': 36848,     # CSEP for graduates within 12 months
    'critical_skills_high_salary': 68911,     # Non-CSOL high-salary route
    'general_employment_standard': 36605,     # Standard GEP
    'general_employment_recent_grad': 34009,  # GEP for Irish-institution recent grads
}

CSL_KEYWORDS = [
    'software engineer', 'software developer', 'data engineer',
    'data scientist', 'machine learning', 'ml engineer', 'devops',
    'site reliability', 'cloud engineer', 'security engineer',
    'cybersecurity', 'backend', 'frontend', 'full stack', 'fullstack',
]

def check_visa_eligibility(offered_salary_eur: float, role_title: str,
                            role_description: str = '',
                            graduated_within_12_months: bool = False) -> dict:
    """
    Determine whether an offer meets visa-permit thresholds (2026 rules).

    Args:
        offered_salary_eur: base salary in EUR
        role_title: job title from the offer / JD
        role_description: optional fuller JD text
        graduated_within_12_months: if True, recent-graduate threshold applies
    """
    text = f'{role_title} {role_description}'.lower()
    likely_csl = any(k in text for k in CSL_KEYWORDS)

    if likely_csl:
        if graduated_within_12_months:
            threshold = VISA_THRESHOLDS_2026['critical_skills_recent_grad']
            permit_type = 'Critical Skills Employment Permit (Recent Graduate route, post-1-March-2026)'
        else:
            threshold = VISA_THRESHOLDS_2026['critical_skills_standard']
            permit_type = 'Critical Skills Employment Permit (Standard route, post-1-March-2026)'
    else:
        if graduated_within_12_months:
            threshold = VISA_THRESHOLDS_2026['general_employment_recent_grad']
            permit_type = 'General Employment Permit (Recent Graduate route, with LMNT)'
        else:
            threshold = VISA_THRESHOLDS_2026['general_employment_standard']
            permit_type = 'General Employment Permit (Standard, with LMNT)'

    headroom = offered_salary_eur - threshold
    meets_threshold = headroom >= 0

    return {
        'offered_salary': offered_salary_eur,
        'role_title': role_title,
        'likely_on_critical_skills_list': likely_csl,
        'graduated_within_12_months': graduated_within_12_months,
        'applicable_threshold': threshold,
        'permit_type': permit_type,
        'headroom_or_shortfall': headroom,
        'meets_threshold': meets_threshold,
        'caveats': [
            'Threshold values reflect post-1-March-2026 rules (DETE Roadmap Review 2025)',
            'Future increases scheduled annually through 2030 with CSO wage indexation',
            'Final occupation classification determined by employer permit application',
            'Employer must independently be willing to sponsor — threshold is necessary not sufficient',
        ],
        'source': 'DETE Roadmap Review 2025, effective 1 March 2026',
    }

# Smoke test
_test = check_visa_eligibility(52000, 'Graduate Software Engineer',
                                'You will write Python and Go services.')
print(f"✓ Visa check: {_test['permit_type']}, threshold {_test['applicable_threshold']}, "
      f"headroom €{_test['headroom_or_shortfall']:,}")

✓ Visa check: Critical Skills Employment Permit (Standard route, post-1-March-2026), threshold 40904, headroom €11,096


---
## Section 4 · Tool Schemas for the Responses API

Tools that the agent can call. For each function we declare a JSON schema so the model knows when and how to invoke it.

In [10]:
TOOLS = [
    # Tool 1 — File Search over the curated knowledge base
    {
        'type': 'file_search',
        'vector_store_ids': [VECTOR_STORE_ID],
    },
    # Tool 2 — Code Interpreter for quantitative computation
    {
        'type': 'code_interpreter',
        'container': {'type': 'auto'},
    },
    # Tool 3 — Function callbacks (one entry per function)
    {
        'type': 'function',
        'name': 'search_dublin_jobs',
        'description': ('Retrieve live job postings for graduate / junior tech '
                        'roles in Dublin from canonical employer applicant-tracking '
                        'systems. Use when the user asks about current openings.'),
        'parameters': {
            'type': 'object',
            'properties': {
                'role_keyword': {
                    'type': 'string',
                    'description': ('keyword to filter job titles, e.g. "graduate", '
                                    '"junior", "data", or empty string for any role')
                },
                'max_results': {
                    'type': 'integer',
                    'description': 'maximum number of jobs to return (1-15)',
                },
            },
            'required': ['role_keyword'],
            'additionalProperties': False,
        },
        'strict': False,
    },
    {
        'type': 'function',
        'name': 'get_user_repos',
        'description': ('Fetch a GitHub user\'s public repositories with metadata, '
                        'used to audit portfolio strength against CV claims.'),
        'parameters': {
            'type': 'object',
            'properties': {
                'username': {'type': 'string', 'description': 'GitHub username'},
                'max_repos': {'type': 'integer', 'description': 'cap on repos fetched'},
            },
            'required': ['username'],
            'additionalProperties': False,
        },
    },
    {
        'type': 'function',
        'name': 'get_repo_readme',
        'description': ('Fetch the README content of a specific GitHub repo, used '
                        'to assess documentation quality of portfolio projects.'),
        'parameters': {
            'type': 'object',
            'properties': {
                'username': {'type': 'string'},
                'repo_name': {'type': 'string'},
            },
            'required': ['username', 'repo_name'],
            'additionalProperties': False,
        },
    },
    {
        'type': 'function',
        'name': 'get_company_insights',
        'description': ('Retrieve current role and culture data for a Dublin-based '
                        'company from The Muse. Supplements static company profiles.'),
        'parameters': {
            'type': 'object',
            'properties': {
                'company_name': {'type': 'string'},
            },
            'required': ['company_name'],
            'additionalProperties': False,
        },
    },
    {
        'type': 'function',
        'name': 'check_visa_eligibility',
        'description': ('Compute whether a specific salary offer meets Irish '
                        'employment-permit thresholds and identify which permit '
                        'type would apply.'),
        'parameters': {
            'type': 'object',
            'properties': {
                'graduated_within_12_months': {
    'type': 'boolean',
    'description': 'whether the user graduated within the previous 12 months (unlocks the lower recent-graduate threshold)',
},
                'offered_salary_eur': {'type': 'number'},
                'role_title': {'type': 'string'},
                'role_description': {'type': 'string'},
            },
            'required': ['offered_salary_eur', 'role_title'],
            'additionalProperties': False,
        },
    },
]

# Map tool names to actual Python callables
FUNCTION_DISPATCH = {
    'search_dublin_jobs': search_dublin_jobs,
    'get_user_repos': get_user_repos,
    'get_repo_readme': get_repo_readme,
    'get_company_insights': get_company_insights,
    'check_visa_eligibility': check_visa_eligibility,
}

print(f'✓ Configured {len([t for t in TOOLS if t["type"] == "function"])} function tools')
print(f'✓ File search + code interpreter enabled')

✓ Configured 5 function tools
✓ File search + code interpreter enabled


---
## Section 5 · System Instructions

These instructions shape the agent's persona, scope, and tool-use heuristics throughout the conversation.

In [11]:
SYSTEM_INSTRUCTIONS = """You are GradPath Dublin, a career-readiness assistant for international Master's-level Computer Science students preparing to apply for graduate / junior software engineering, data, or ML roles in Dublin, Ireland.

## Your user

The user is typically:
- A non-EU/EEA Master's student at a Dublin-area university (UCD, Trinity, DCU, etc.)
- On Stamp 2 (student) and planning to use Stamp 1G post-graduation
- Targeting employers in the Dublin tech market
- A non-native English speaker, often Chinese, navigating cross-cultural workplace norms for the first time

## Your scope

You help with: CV diagnostics, GitHub portfolio audits, live job-market awareness, visa-pathway feasibility, and interview preparation including cross-cultural communication.

You DO NOT provide legal immigration advice. Anything visa-related is educational reference, and you must always direct the user to authoritative sources (INIS, DETE) for decisions.

## How to use your tools

You have three tool categories. Use them as follows:

**File Search** (your knowledge base):
- Use to retrieve company-specific information (tech stack, hiring process)
- Use to retrieve interview patterns, scoring rubrics, visa policy summaries, cultural-norm guidance
- Always use file search for these topics rather than relying on general knowledge

**Function Calls** (external APIs):
- `search_dublin_jobs` — when the user asks about current job openings
- `get_user_repos` and `get_repo_readme` — when the user provides a GitHub username and wants their portfolio assessed
- `get_company_insights` — when the user asks for up-to-date insight on a specific company beyond what's in the knowledge base
- `check_visa_eligibility` — when the user mentions a specific salary and asks about visa feasibility

**Tool synthesis is essential**:
When you retrieve live job postings via search_dublin_jobs, you should ALWAYS
follow up with file_search on the relevant company knowledge (search the
vector store with the company name + "Dublin"). The job listing tells you
what's open; the knowledge base tells you the company's interview style,
tech stack, sponsorship history, and how to prioritise. A good answer
combines both. Never give priority recommendations using only the live job
data — always enrich with the curated company profiles.

**Code Interpreter**:
- Use for CV-to-JD match scoring (TF-IDF or keyword overlap)
- Use for any quantitative comparison or visualisation
- Use to compute composite Career-Readiness scores from sub-scores
- Do NOT use for things you can answer directly — code interpreter is the most expensive tool



## How you should respond

- Be specific and concrete. Vague encouragement is unhelpful.
- When you give recommendations, name the next concrete action and why it matters.
- When using rubrics from the knowledge base, show the breakdown — don't just give a single score.
- For Chinese students: be aware of common cultural pitfalls (modesty erasure, indirectness, missing work-authorisation). Surface these when relevant.
- Always cite which file from the knowledge base or which API call your information came from when relevant.
- When uncertain, say so. Never fabricate salary figures, policy details, or company specifics.
"""



print(f'✓ System instructions configured ({len(SYSTEM_INSTRUCTIONS)} chars)')

✓ System instructions configured (3235 chars)


---
## Section 6 · Conversation Loop

The core agentic loop: send user message → receive tool calls / text → execute tools → feed results back → continue until model returns final text.

In [12]:
def run_agent_turn(conversation_history: list, user_message: str,
                   max_tool_iterations: int = 8, verbose: bool = True) -> tuple:
    """
    Run one user turn through the agent. Handles tool-call loop until the
    model returns final text.

    Returns: (final_text, updated_conversation_history)
    """
    # Append user message
    conversation_history = conversation_history + [
        {'role': 'user', 'content': user_message}
    ]

    iteration = 0
    while iteration < max_tool_iterations:
        iteration += 1
        if verbose:
            print(f'\n  [iter {iteration}] calling model...')

        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=conversation_history,
            tools=TOOLS,
            temperature=0.4,
        )

        # Did the model request any function calls?
        function_calls = [item for item in response.output
                          if getattr(item, 'type', None) == 'function_call']

        if not function_calls:
            # No function calls — final answer
            if verbose:
                print(f'  [iter {iteration}] model returned final answer')
            # Append all output items to history for context continuity
            for item in response.output:
                conversation_history.append(item)
            return response.output_text, conversation_history

        # Execute each function call and append the result
        if verbose:
            print(f'  [iter {iteration}] model requested {len(function_calls)} function call(s)')
        # Append the model's call items to history
        for item in response.output:
            conversation_history.append(item)
        # Then append each function output
        for fc in function_calls:
            fname = fc.name
            fargs = json.loads(fc.arguments)
            if verbose:
                print(f'      → {fname}({fargs})')
            if fname not in FUNCTION_DISPATCH:
                result = {'error': f'unknown function: {fname}'}
            else:
                try:
                    result = FUNCTION_DISPATCH[fname](**fargs)
                except Exception as e:
                    result = {'error': str(e)}
            conversation_history.append({
                'type': 'function_call_output',
                'call_id': fc.call_id,
                'output': json.dumps(result, default=str),
            })

    # Safety exit
    return '⚠ Reached max tool iterations without final answer', conversation_history

print('✓ Agent loop function defined')

✓ Agent loop function defined


---
## Section 7 · Worked Example 1 · Live job market scan

**Scenario**: A user asks for current Dublin graduate-level engineering openings.

**Expected tool path**: function call (`search_dublin_jobs`) → file search (company profiles) → final synthesis.

In [13]:
conversation = []

user_msg_1 = (
    "I'm a Master's student at UCD finishing in June, looking for graduate "
    "software engineering or junior data engineer roles in Dublin. What's "
    "open right now, and which ones should I prioritise?"
)

print('USER:')
print(user_msg_1)
print('\n' + '='*60)
print('AGENT:')
print('='*60)

answer_1, conversation = run_agent_turn(conversation, user_msg_1)
print('\n' + answer_1)

USER:
I'm a Master's student at UCD finishing in June, looking for graduate software engineering or junior data engineer roles in Dublin. What's open right now, and which ones should I prioritise?

AGENT:

  [iter 1] calling model...
  [iter 1] model requested 1 function call(s)
      → search_dublin_jobs({'role_keyword': 'graduate', 'max_results': 10})


/tmp/ipykernel_27543/1091778175.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'fetched_at': datetime.utcnow().isoformat() + 'Z',



  [iter 2] calling model...
  [iter 2] model requested 1 function call(s)
      → search_dublin_jobs({'role_keyword': 'junior', 'max_results': 10})

  [iter 3] calling model...
  [iter 3] model requested 1 function call(s)
      → search_dublin_jobs({'role_keyword': 'data', 'max_results': 10})

  [iter 4] calling model...
  [iter 4] model returned final answer

Here’s a summary of current graduate/junior software/data roles open in Dublin, and how you should prioritise them as a UCD Master's student finishing in June:

---

## 1. **Live Openings (May 2026)**
From top Dublin tech employers, these data/software roles are currently open:

- **Stripe**  
  - [Data Analyst](https://stripe.com/jobs/search?gh_jid=7610132)  
  - [Data Analyst, Financial Enablement](https://stripe.com/jobs/search?gh_jid=7874654)
- **Intercom**  
  - [Growth Data Scientist](https://job-boards.greenhouse.io/intercom/jobs/7672152)  
  - [Senior Data Scientist - AI Tooling](https://job-boards.greenhouse.io/interco

---
## Section 8 · Worked Example 2 · GitHub portfolio audit

**Scenario**: User wants their GitHub portfolio assessed against grad-SWE expectations.

**Expected tool path**: function calls (`get_user_repos`, possibly `get_repo_readme`) → file search (`rubric_github_portfolio_audit.md`) → code interpreter (composite scoring) → synthesis.

We use `octocat` as a public test username — replace with a real candidate username when running live.

In [33]:
# Fresh conversation for a clean example
conversation_2 = []

user_msg_2 = (
    "My GitHub username is `octocat`. Can you audit my public portfolio "
    "and tell me whether it would support an application for a graduate "
    "Software Engineer role at a Dublin tech employer? Please use a "
    "rubric and give me concrete improvements."
)

print('USER:')
print(user_msg_2)
print('\n' + '='*60)
print('AGENT:')
print('='*60)

answer_2, conversation_2 = run_agent_turn(conversation_2, user_msg_2)
print('\n' + answer_2)

USER:
My GitHub username is `octocat`. Can you audit my public portfolio and tell me whether it would support an application for a graduate Software Engineer role at a Dublin tech employer? Please use a rubric and give me concrete improvements.

AGENT:

  [iter 1] calling model...
  [iter 1] model requested 1 function call(s)
      → get_user_repos({'username': 'octocat', 'max_repos': 10})

  [iter 2] calling model...
  [iter 2] model returned final answer

Here’s a detailed audit of your public GitHub portfolio (`octocat`) using the Dublin graduate Software Engineer rubric. This is based on your most visible 8 public repositories. If you want a deeper repo-by-repo breakdown or want me to check specific repos, let me know!

---

## Portfolio Audit (Out of 100)

### A. Pinned Repository Quality (25)
- You have several high-star, high-fork repos, but many are demo/test or forks, not original, substantive projects.
- Most are not curated for relevance to SWE roles (e.g., `Spoon-Knife` is 

---
## Section 9 · Worked Example 3 · Visa-feasibility for a specific offer

**Scenario**: User has an offer in hand and wants to know whether it qualifies for visa sponsorship.

**Expected tool path**: function call (`check_visa_eligibility`) → file search (`visa_*` files) → synthesis with caveats.

In [34]:
conversation_3 = []

user_msg_3 = (
    "I have an offer for a Junior Data Engineer role at a Dublin company. "
    "Base salary is €42,000. The role description mentions Python, SQL, "
    "Apache Airflow, and AWS. Will I be able to get a work permit on this "
    "offer? What should I check before accepting?"
)

print('USER:')
print(user_msg_3)
print('\n' + '='*60)
print('AGENT:')
print('='*60)

answer_3, conversation_3 = run_agent_turn(conversation_3, user_msg_3)
print('\n' + answer_3)

USER:
I have an offer for a Junior Data Engineer role at a Dublin company. Base salary is €42,000. The role description mentions Python, SQL, Apache Airflow, and AWS. Will I be able to get a work permit on this offer? What should I check before accepting?

AGENT:

  [iter 1] calling model...
  [iter 1] model requested 1 function call(s)
      → check_visa_eligibility({'graduated_within_12_months': True, 'offered_salary_eur': 42000, 'role_title': 'Junior Data Engineer', 'role_description': 'Python, SQL, Apache Airflow, AWS'})

  [iter 2] calling model...
  [iter 2] model returned final answer

Based on your offer details:

- Role: Junior Data Engineer (Python, SQL, Apache Airflow, AWS)
- Salary: €42,000
- You are a recent graduate (within 12 months)

Here’s what you need to know:

### 1. Work Permit Eligibility

- Your role is likely on the Critical Skills Occupations List (Data Engineer roles are typically included).
- As a recent graduate, the minimum salary threshold for a Critical S

---
## Section 10 · Cleanup (optional)

Run this only if you want to delete the vector store after evaluation. Leaving it in place is fine — Responses-API vector stores are not billed per-day under the current pricing model.

In [ ]:
# Uncomment to delete
# client.vector_stores.delete(VECTOR_STORE_ID)
# os.remove(VECTOR_STORE_CACHE)
# print('✓ Vector store deleted')


NameError: name 'client' is not defined